# Causal explanation of drift

In this Python notebook, we analyze causal machine learning methods for drift explanation. Therefore, we load a dataset with a causal ground truth provided by a DAG. We use the PC algorithm to learn a causal ground truth and compare it with the predicted causal ground truth.

We compare the results by visualizing the difference between the average prediction and the causal ground truth.

The data can be downloaded from [[this github repo](https://github.com/DKomnick/Streaming-data-causality)]. 

In [ ]:
import numpy as np
import pandas as pd
import glob

from causallearn.search.ConstraintBased import PC
import causallearn.utils.cit as cit
from causallearn.graph.Edge import Edge
from causallearn.graph.Endpoint import Endpoint
from causallearn.graph.GraphClass import CausalGraph

We create a causal ground truth and prepare the features.

In [ ]:
# local variables, configure them for yourself

base_path = f'/media/local/dkomnick/data_causalExplanationOfDrift/data/'

student_feature = ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G3', 'T']

adult_features = ['age', 'workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
       'hours-per-week', 'native-country', 'income', 'T']


student_dirs = ['unmodified',
           'unmodified_to_females_more_support_2000',
           'unmodified_to_grade_category_adjusted_2000',
           'unmodified_to_grade_inflation_2000',
           'unmodified_to_internet_era_2000',
           'unmodified_to_males_more_support_2000']

adult_dirs = ['unmodified',
         'unmodified_to_less_marriage_25000',
         'unmodified_to_men_work_full_25000',
         'unmodified_to_men_work_more_25000',
         'unmodified_to_women_stem_25000',
         'unmodified_to_inflation_25000',
         'unmodified_to_women_work_full_25000',
         'unmodified_to_women_work_more_25000']


In [ ]:
# create a causal ground truth. Note, that the variables "t_adult" and "children" (from the function ground_truth_student) depend on the order "student_dirs" and "adult_dirs".
t_adult = [[], [3], [10], [10], [4], [8, 9, 12], [10], [10]]

ground_truth_adult = [[ 0, -1, -1, -1, -1,  1, -1,  0,  0,  0, -1, -1, -1],
                [ 1,  0, -1,  0, -1,  1,  0,  0,  0,  0,  1, -1,  0],
                [ 1,  1,  0,  0,  1,  0,  0,  1, -1,  0,  0, -1, -1],
                [ 1,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  1,  0],
                [ 1,  1, -1,  0,  0,  0,  0,  1, -1, -1,  1,  0, -1],
                [-1, -1,  0, -1,  0,  0, -1,  1, -1, -1, -1,  0, -1],
                [ 1,  0,  0,  0,  0,  1,  0,  1,  0,  0,  0,  1,  0],
                [ 0,  0, -1,  0, -1, -1, -1,  0,  0,  0, -1,  0,  0],
                [ 0,  0,  1,  0,  1,  1,  0,  0,  0, -1,  0,  0, -1],
                [ 0,  0,  0,  0,  1,  1,  0,  0,  1,  0,  0,  0, -1],
                [ 1, -1,  0,  0, -1,  1,  0,  1,  0,  0,  0,  0,  0],
                [ 1,  1,  1, -1,  0,  0, -1,  0,  0,  0,  0,  0,  0],
                [ 1,  0,  1,  0,  1,  1,  0,  0,  1,  1,  0,  0,  0]]
ground_truth_adult = np.array(ground_truth_adult).T

def ground_truth_student_() -> np.array:
    children = [('school', ['higher', 'internet', 'address', 'reason', 'failures', 'G3']),
            ('famsup', ['sex']),
            ('activities', ['sex']),
            ('sex', ['higher', 'failures', 'schoolsup', 'G3']),
            ('schoolsup', ['school']),
            ('address', ['traveltime', 'internet']),
            ('internet', ['Mjob']),
            ('Mjob', ['Medu']),
            ('Medu', ['Fedu']),
            ('Fedu', ['Fjob']),
            ('failures', ['age']),
            ('age', ['higher', 'romantic', 'guardian']),
            ('guardian', ['Pstatus']),
            ('Pstatus', ['famsize']),
            ('famsize', ['nursery'])]
    
    dic_stud = dict()
    for i, s in enumerate(student_feature[:-1]):
        dic_stud[s] = i

    arr = np.zeros((len(student_feature[:-1]), len(student_feature[:-1])))

    for par, l_chil in children:
        for c in l_chil:
            arr[dic_stud[par], dic_stud[c]] = 1
            arr[dic_stud[c], dic_stud[par]] = -1
    return arr, [[], [dic_stud['schoolsup']], [dic_stud['G3']], [dic_stud['G3'], dic_stud['failures'], dic_stud['higher']], [dic_stud['internet']], [dic_stud['schoolsup']]]

ground_truth_student, t_student = ground_truth_student_()

In [ ]:
# run the PC algorithm

cgs_student = []
for dir in student_dirs:
    l = []
    for file in glob.glob(base_path + 'student_performance/' + dir + '/*.csv'):
        with open(file) as f:
            df = pd.read_csv(f)
        df['T'] = 0
        if file != 'unmodified':
            df.loc[2000:, 'T'] = 1
        df = df[student_feature]
        for col in df.columns:
            dic = dict()
            for i, k in enumerate(set(df[col])):
                dic[k] = i
            df[col] = df[col].map(lambda x: dic[x])
        cg = PC.pc(df.to_numpy(), 0.01, cit.gsq)
        l.append(cg)
    cgs_student.append(l)

cgs_adult = []
for dir in adult_dirs:
    l = []
    for file in glob.glob(base_path + 'adult/' + dir + '/*.csv'):
        with open(file) as f:
            df = pd.read_csv(f)
        df['T'] = 0
        # as a dataset-artifact, the "unnamed" column exists. Remove it by making it the index
        if 'Unnamed: 0.1' in df.columns:
            df.set_index('Unnamed: 0.1')
        if file != 'unmodified':
            df.loc[25000:, 'T'] = 1
        df = df[adult_features]
        for col in df.columns:
            dic = dict()
            for i, k in enumerate(set(df[col])):
                dic[k] = i
            df[col] = df[col].map(lambda x: dic[x])
        cg = PC.pc(df.to_numpy(), 0.01, cit.gsq)
        l.append(cg)
    cgs_adult.append(l)


In [ ]:
# a function creating the information needed for a graph based on the 

def array2dag(array):
    num_nodes = array.shape[0]
    cg = CausalGraph(num_nodes)
    for i in range(num_nodes):
        for j in range(num_nodes):
            edge1 = cg.G.get_edge(cg.G.nodes[i], cg.G.nodes[j])
            if edge1 is not None:
                cg.G.remove_edge(edge1)

    for i in range(num_nodes):
        for j in range(num_nodes):
            if array[i,j] == 1:
                cg.G.add_edge(Edge(cg.G.nodes[i], cg.G.nodes[j], Endpoint.TAIL, Endpoint.ARROW))

    DAG = cg
    return DAG

def plot_cg(cgs, gt :np.array, t_connected, features):
    # add T-node to feature
    gt_ = np.zeros((gt.shape[0] + 1, gt.shape[0] + 1))
    gt_[:-1,:-1] = gt
    gt_[-1, t_connected] = 1
    gt_[t_connected, -1] = -1
    gt = array2dag(gt_)

    # plot graph
    for cg in cgs:
        assert len(cg.G.nodes) == len(gt.G.nodes), "Not all graphs have the same amount of nodes!"

    edges = []
    
    # iterate over all possible edges
    for i in range(len(gt.G.nodes)):
        for j in range(i + 1, len(gt.G.nodes)):
            count = 0
            l2h, h2l = 0, 0 # "lower to higher" and "higher to lower" edge directed to the lower or higher node orientation
            in_gt = False
            for k, cg in enumerate([gt] + cgs):
                old_count = count
                for c in cg.G.get_graph_edges():
                    assert c.get_endpoint1() in [Endpoint.TAIL, Endpoint.ARROW]
                    assert c.get_endpoint2() in [Endpoint.TAIL, Endpoint.ARROW]
                    if (int(c.node1.name[1:]) == i + 1 and int(c.node2.name[1:]) == j + 1):
                        if k == 0:
                            in_gt = 'l2h' if c.get_endpoint2() == Endpoint.ARROW else 'h2l'
                        else:
                            count += 1
                            h2l += int(c.get_endpoint1() == Endpoint.ARROW)
                            l2h += int(c.get_endpoint2() == Endpoint.ARROW)
                    elif (int(c.node1.name[1:]) == j + 1 and int(c.node2.name[1:]) == i + 1):
                        if k == 0:
                            in_gt = 'l2h' if c.get_endpoint1() == Endpoint.ARROW else 'h2l'
                        else:
                            count += 1
                            l2h += int(c.get_endpoint1() == Endpoint.ARROW)
                            h2l += int(c.get_endpoint2() == Endpoint.ARROW)
                assert (count - old_count) <= 1 # edge is only found once

            count = float(count)
            n_graphs = float(len(cgs))
            if count != 0 or in_gt != False:
                if in_gt != False:
                    in_gt = (features[i],features[j]) if in_gt == 'l2h' else (features[j],features[i])
                    count -= 1
                    n_graphs -= 1
                if l2h == h2l:
                    # undirected
                    edges.append(((features[j], features[i]), count/n_graphs, in_gt))
                elif h2l < l2h:
                    # lower2higher 
                    edges.append(((features[i], features[j]), count/n_graphs, in_gt))
                else:
                    # higher2lower
                    edges.append(((features[j], features[i]), count/n_graphs, in_gt))

    return edges

In [ ]:
results_information_adult = dict()

for i, l in enumerate(cgs_adult):
    edges = plot_cg(l, ground_truth_adult, t_adult[i], adult_features)
    results_information_adult[adult_dirs[i]] = edges

results_information_student = dict()
for i, l in enumerate(cgs_student):
    edges = plot_cg(l, ground_truth_student, t_student[i], student_feature)
    results_information_student[student_dirs[i]] = edges

`results_information_adult` and `results_information_student` contain all information needed to plot the graphs. The format of each entry is
`((source, dest), proportion_hits, (source_ground_truth, dest_ground_truth))`